# Tool Usage Diversity

The Tool Usage Diversity notebook contains the implementation, tests, and usage workflow for the Tool Usage Diversity metric. It scans JSON/JSONL conversation traces, counts structured assistant tool calls, and calculates normalized entropy against one fixed tool inventory.

The score is `H / ln(K)`, where `K` is the number of configured tools. A reference tool with no calls remains in the denominator. No recognized calls means the score is `None` (undefined). Unknown tools are reported and excluded from scoring. A one-tool inventory scores `0.0` when it has calls. The score measures the aggregate tool-call distribution, not equality of usage on every task.

The parser handles structured `tool_calls` and legacy `function_call` fields on assistant/AI messages. It does not infer calls from text or tool results. The supplied tau2 `results.json` layout (`simulations` with `messages`) has been checked; other benchmark output formats may need a parser adjustment.

To run: open this notebook, run the cells from top to bottom, edit the configuration in the final cell, then run that cell. The tests use only Python's standard library. You will need to have ran tau-bench as well as know the paths to the input and output of your results.

In [ ]:
from collections import Counter
from pathlib import Path
from typing import Any
import json
import math


def tool_name(call: Any) -> str | None:
    """Get a tool name from a common structured tool-call object."""
    if not isinstance(call, dict):
        return None
    name = call.get("name")
    if not isinstance(name, str) and isinstance(call.get("function"), dict):
        name = call["function"].get("name")
    return name.strip() if isinstance(name, str) and name.strip() else None


def extract_tool_calls(record: Any) -> list[str]:
    """Extract structured calls from one message list or conversation record."""
    if isinstance(record, list):
        messages = record
    elif isinstance(record, dict):
        messages = next(
            (record[key] for key in ("messages", "conversation", "trajectory")
             if isinstance(record.get(key), list)),
            None,
        )
        if messages is None and record.get("role", record.get("type")) in {
            "assistant", "ai", "AIMessage"
        }:
            messages = [record]
    else:
        return []

    if not isinstance(messages, list):
        return []

    calls, seen_ids = [], set()
    for message in messages:
        if not isinstance(message, dict):
            continue
        payload = message.get("data", message)
        if not isinstance(payload, dict):
            continue
        role = str(message.get("role", message.get("type", ""))).lower()
        if role not in {"assistant", "ai", "aimessage"}:
            continue

        raw_calls = payload.get("tool_calls")
        if not isinstance(raw_calls, list):
            extra = payload.get("additional_kwargs", {})
            raw_calls = extra.get("tool_calls") if isinstance(extra, dict) else None
        if not isinstance(raw_calls, list):
            legacy = payload.get("function_call")
            raw_calls = [legacy] if isinstance(legacy, dict) else []

        for call in raw_calls:
            name = tool_name(call)
            if name is None:
                continue
            call_id = call.get("id") if isinstance(call, dict) else None
            if call_id and call_id in seen_ids:
                continue
            if call_id:
                seen_ids.add(call_id)
            calls.append(name)
    return calls


def calculate_diversity(call_counts: dict[str, int], reference_tools: list[str]) -> dict[str, Any]:
    """Calculate entropy and report counts for a fixed reference inventory."""
    tools = sorted(set(reference_tools))
    if not tools:
        raise ValueError("Configure at least one reference tool.")
    if any(not isinstance(n, int) or isinstance(n, bool) or n < 0 for n in call_counts.values()):
        raise ValueError("Tool counts must be non-negative integers.")

    counts = {name: call_counts.get(name, 0) for name in tools}
    unknown = {name: count for name, count in sorted(call_counts.items())
               if name not in tools and count > 0}
    total = sum(counts.values())
    entropy = -sum((count / total) * math.log(count / total)
                   for count in counts.values() if count) if total else None
    score = (0.0 if len(tools) == 1 else max(0.0, min(1.0, entropy / math.log(len(tools))))) if total else None
    used = sum(count > 0 for count in counts.values())
    balance = (0.0 if used <= 1 else entropy / math.log(used)) if total else None

    return {
        "total_tool_calls": total + sum(unknown.values()),
        "recognized_tool_calls": total,
        "unknown_tool_calls": sum(unknown.values()),
        "unknown_tool_call_counts": unknown,
        "distinct_tools_used": used,
        "reference_tool_count": len(tools),
        "tool_call_counts": counts,
        "tool_call_proportions": {name: count / total if total else 0.0
                                   for name, count in counts.items()},
        "raw_shannon_entropy": entropy,
        "tool_balance": balance,
        "normalized_tool_usage_diversity": score,
    }


